CNNベースのやつ

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, models, transforms
from torch.utils.data import DataLoader, Subset  # 【修正1】Subsetを追加
import os
from collections import Counter
import numpy as np

def main():
    # -----------------------------
    # 1. 設定パラメータ
    # -----------------------------
    DATA_DIR = '/home/data/1021_fasttest/crop'
    BATCH_SIZE = 32
    NUM_EPOCHS = 10
    LEARNING_RATE = 0.001
    NUM_CLASSES = 3

    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    print(f"使用デバイス: {device}")

    # -----------------------------
    # 2. データの前処理と読み込み
    # -----------------------------
    data_transforms = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])

    full_dataset = datasets.ImageFolder(DATA_DIR, transform=data_transforms)
    class_names = full_dataset.classes
    print(f"検知されたクラス: {class_names}") 

    # ---------------------------------------------------------
    # データ分割処理 (クラスごとに指定枚数を抽出)
    # ---------------------------------------------------------
    target_counts = {
        0: 34,  # A
        1: 33,  # B
        2: 33   # C
    }
    
    train_indices = []
    val_indices = []
    
    all_targets = np.array(full_dataset.targets)
    np.random.seed(42) # シード固定

    print("\n=== データ分割処理 ===")
    
    for class_idx in range(NUM_CLASSES):
        # そのクラスに該当する画像のインデックスを全て取得
        indices_of_class = np.where(all_targets == class_idx)[0]
        
        # インデックスをシャッフル
        np.random.shuffle(indices_of_class)
        
        # 指定枚数を取得
        count = target_counts[class_idx]
        
        # データ数が足りているかチェック
        if len(indices_of_class) < count:
             raise ValueError(f"クラス {class_names[class_idx]} のデータ不足: {len(indices_of_class)}枚しかありませんが {count}枚 要求されています。")

        # リストに追加
        train_indices.extend(indices_of_class[:count])
        val_indices.extend(indices_of_class[count:])
        
        print(f"クラス {class_names[class_idx]}: 学習用 {count}枚 / 検証用 {len(indices_of_class) - count}枚")

    # Subsetを使ってデータセットを作成
    train_dataset = Subset(full_dataset, train_indices)
    val_dataset = Subset(full_dataset, val_indices)

    # ---------------------------------------------------------
    # ▼ 追加: 分割情報（インデックス）を保存する
    # ---------------------------------------------------------
    split_info = {
        'train_indices': train_indices,
        'val_indices': val_indices,
        'class_names': class_names
    }
    torch.save(split_info, 'dataset_split.pth')
    print("データセットの分割情報を 'dataset_split.pth' に保存しました。")

    # 【修正2】train_size, val_size を定義しなおす
    train_size = len(train_dataset)
    val_size = len(val_dataset)

    # 念のため内訳確認
    print("\n=== 最終確認: 学習データ内訳 ===")
    train_labels_check = [full_dataset.targets[i] for i in train_indices]
    print(Counter(train_labels_check)) 
    print("===============================\n")

    # DataLoaderの作成
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

    # -----------------------------
    # 3. モデルの構築 (ResNet50)
    # -----------------------------
    model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
    num_ftrs = model.fc.in_features
    model.fc = nn.Linear(num_ftrs, NUM_CLASSES)
    model = model.to(device)

    # -----------------------------
    # 4. 損失関数と最適化手法
    # -----------------------------
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

    # -----------------------------
    # 5. 学習ループ
    # -----------------------------
    print("学習を開始します...")
    
    for epoch in range(NUM_EPOCHS):
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0

        # --- Training ---
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * inputs.size(0)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

        epoch_loss = running_loss / train_size
        epoch_acc = correct / total

        # --- Validation ---
        model.eval()
        val_loss = 0.0
        val_correct = 0
        val_total = 0
        
        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                
                val_loss += loss.item() * inputs.size(0)
                _, predicted = torch.max(outputs, 1)
                val_total += labels.size(0)
                val_correct += (predicted == labels).sum().item()

        val_epoch_loss = val_loss / val_size
        val_epoch_acc = val_correct / val_total

        print(f'Epoch {epoch+1}/{NUM_EPOCHS} '
              f'Train Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f} | '
              f'Val Loss: {val_epoch_loss:.4f} Acc: {val_epoch_acc:.4f}')

    # -----------------------------
    # 6. モデルの保存
    # -----------------------------
    save_path = 'resnet50_100.pth'
    torch.save(model.state_dict(), save_path)
    print(f"学習完了。モデルを保存しました: {save_path}")

if __name__ == '__main__':
    main()

In [ ]:
import torch
import torch.nn as nn
from torchvision import datasets, models, transforms
from torch.utils.data import DataLoader, Subset
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import os

def main():
    # -----------------------------
    # 1. 設定
    # -----------------------------
    DATA_DIR = '/home/data/1021_fasttest/combined'
    MODEL_PATH = 'resnet50_100.pth'      # 学習済みモデル
    SPLIT_PATH = 'dataset_split.pth'     # ★保存した分割情報
    BATCH_SIZE = 32
    NUM_CLASSES = 3
    
    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    print(f"デバイス: {device}")

    # -----------------------------
    # 2. 分割情報の読み込み
    # -----------------------------
    if not os.path.exists(SPLIT_PATH):
        print(f"エラー: {SPLIT_PATH} が見つかりません。学習時に保存しましたか？")
        return

    # 保存した辞書をロード
    split_info = torch.load(SPLIT_PATH,weights_only=False)
    val_indices = split_info['val_indices']     # 検証用に割り当てられたインデックス
    class_names = split_info['class_names']     # クラス名
    
    print(f"分割情報をロードしました。検証用データ数: {len(val_indices)}枚")

    # -----------------------------
    # 3. データセットの再構築
    # -----------------------------
    data_transforms = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])

    full_dataset = datasets.ImageFolder(DATA_DIR, transform=data_transforms)

    # ロードしたインデックスを使って、検証用データセット(Subset)を復元
    val_dataset = Subset(full_dataset, val_indices)

    # DataLoader作成
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

    # -----------------------------
    # 4. モデルロードと推論
    # -----------------------------
    model = models.resnet50(weights=None)
    num_ftrs = model.fc.in_features
    model.fc = nn.Linear(num_ftrs, NUM_CLASSES)

    model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
    model = model.to(device)
    model.eval()

    all_preds = []
    all_labels = []

    print("推論を実行中...")
    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    # -----------------------------
    # 5. 結果表示
    # -----------------------------
    acc = accuracy_score(all_labels, all_preds)
    print(f"\n=== テスト結果 (保存された検証セットを使用) ===")
    print(f"Accuracy: {acc:.4f} ({acc*100:.2f}%)")
    
    print("\n=== 詳細レポート ===")
    print(classification_report(all_labels, all_preds, target_names=class_names))

    # 混同行列
    cm = confusion_matrix(all_labels, all_preds)
    cm_df = pd.DataFrame(cm, index=class_names, columns=class_names)
    
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm_df, annot=True, fmt='d', cmap='Blues')
    plt.title('Confusion Matrix (Validation Set)')
    plt.ylabel('True')
    plt.xlabel('Pred')
    plt.tight_layout()
    plt.savefig('test_result_matrix.png')
    print("混同行列を 'test_result_matrix.png' に保存しました。")

if __name__ == '__main__':
    main()

ループ

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, models, transforms
from torch.utils.data import DataLoader, Subset
import os
from collections import Counter
import numpy as np
import math

def main():
    # -----------------------------
    # 1. 設定パラメータ
    # -----------------------------
    DATA_DIR = '/home/data/1021_fasttest/crop' # フォルダ名は環境に合わせて確認してください
    BATCH_SIZE = 8
    NUM_EPOCHS = 50
    LEARNING_RATE = 0.001
    NUM_CLASSES = 3
    
    # 学習させたいデータ総数のリスト
    TRAIN_SIZES = [50,60,70,80,90, 100,110,120,130,140, 150, 160,170,180,190,200]

    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    print(f"使用デバイス: {device}")

    # -----------------------------
    # 2. データの前処理と読み込み (共通)
    # -----------------------------
    data_transforms = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])

    full_dataset = datasets.ImageFolder(DATA_DIR, transform=data_transforms)
    class_names = full_dataset.classes
    all_targets = np.array(full_dataset.targets)
    
    print(f"検知されたクラス: {class_names}") 

    # =========================================================
    #  ここからサイズごとのループ実行
    # =========================================================
    for total_train_size in TRAIN_SIZES:
        print(f"\n##################################################")
        print(f"  開始: 学習データ数 {total_train_size} 枚のパターン")
        print(f"##################################################")

        # ---------------------------------------------------------
        # クラスごとの枚数を計算 (均等割り振り)
        # ---------------------------------------------------------
        # 例: 50枚 -> base=16, remainder=2 -> [17, 17, 16]
        base_count = total_train_size // NUM_CLASSES
        remainder = total_train_size % NUM_CLASSES
        
        target_counts = {}
        for i in range(NUM_CLASSES):
            # 余りがある場合は、前のクラスから順に+1する
            add = 1 if i < remainder else 0
            target_counts[i] = base_count + add

        print(f"目標内訳: {target_counts}")

        # ---------------------------------------------------------
        # データ分割処理
        # ---------------------------------------------------------
        train_indices = []
        val_indices = []
        
        # 毎回シードを固定しなおす（「100枚」のデータが「50枚」のデータを含むようにするため）
        np.random.seed(42)

        for class_idx in range(NUM_CLASSES):
            indices_of_class = np.where(all_targets == class_idx)[0]
            np.random.shuffle(indices_of_class) # シャッフル
            
            count = target_counts[class_idx]
            
            # データ数チェック
            if len(indices_of_class) < count:
                 raise ValueError(f"クラス {class_names[class_idx]} データ不足: 保有{len(indices_of_class)} < 要求{count}")

            train_indices.extend(indices_of_class[:count])
            val_indices.extend(indices_of_class[count:])
        
        # Dataset作成
        train_dataset = Subset(full_dataset, train_indices)
        val_dataset = Subset(full_dataset, val_indices)

        # ---------------------------------------------------------
        # 分割情報の保存 (ファイル名を動的に変更)
        # ---------------------------------------------------------
        split_filename = f'res18_crop/dataset_split_{total_train_size}.pth'
        split_info = {
            'train_indices': train_indices,
            'val_indices': val_indices,
            'class_names': class_names,
            'total_size': total_train_size
        }
        torch.save(split_info, split_filename)
        print(f"分割情報を保存: {split_filename}")

        train_size_real = len(train_dataset)
        val_size_real = len(val_dataset)
        
        print(f"学習データ内訳: {Counter([full_dataset.targets[i] for i in train_indices])}")

        # DataLoader作成
        train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
        val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

        # -----------------------------
        # 3. モデルの構築 (毎回初期化！)
        # -----------------------------
        model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        num_ftrs = model.fc.in_features
        model.fc = nn.Linear(num_ftrs, NUM_CLASSES)
        model = model.to(device)

        # -----------------------------
        # 4. 損失関数と最適化手法
        # -----------------------------
        criterion = nn.CrossEntropyLoss()
        optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

        # -----------------------------
        # 5. 学習ループ
        # -----------------------------
        for epoch in range(NUM_EPOCHS):
            model.train()
            running_loss = 0.0
            correct = 0
            total = 0

            # Training
            for inputs, labels in train_loader:
                inputs, labels = inputs.to(device), labels.to(device)

                optimizer.zero_grad()
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                loss.backward()
                optimizer.step()

                running_loss += loss.item() * inputs.size(0)
                _, predicted = torch.max(outputs, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()

            epoch_loss = running_loss / train_size_real
            epoch_acc = correct / total

            # Validation
            model.eval()
            val_loss = 0.0
            val_correct = 0
            val_total = 0
            
            with torch.no_grad():
                for inputs, labels in val_loader:
                    inputs, labels = inputs.to(device), labels.to(device)
                    outputs = model(inputs)
                    loss = criterion(outputs, labels)
                    
                    val_loss += loss.item() * inputs.size(0)
                    _, predicted = torch.max(outputs, 1)
                    val_total += labels.size(0)
                    val_correct += (predicted == labels).sum().item()

            val_epoch_loss = val_loss / val_size_real
            val_epoch_acc = val_correct / val_total
            
            # ログは少し簡略化して表示
            print(f'[{total_train_size}枚パターン] Epoch {epoch+1}/{NUM_EPOCHS} '
                  f'Train Acc: {epoch_acc:.4f} | Val Acc: {val_epoch_acc:.4f}')

        # -----------------------------
        # 6. モデルの保存 (ファイル名を動的に変更)
        # -----------------------------
        model_filename = f'res18_crop/resnet18_{total_train_size}.pth'
        torch.save(model.state_dict(), model_filename)
        print(f"★ モデル保存完了: {model_filename}\n")

    print("全てのパターンの学習が完了しました。")

if __name__ == '__main__':
    main()

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


使用デバイス: cpu
検知されたクラス: ['A', 'B', 'C']

##################################################
  開始: 学習データ数 50 枚のパターン
##################################################
目標内訳: {0: 17, 1: 17, 2: 16}
分割情報を保存: res18_crop/dataset_split_50.pth
学習データ内訳: Counter({0: 17, 1: 17, 2: 16})


100%|██████████| 44.7M/44.7M [00:00<00:00, 104MB/s] 


[50枚パターン] Epoch 1/50 Train Acc: 0.5200 | Val Acc: 0.6154
[50枚パターン] Epoch 2/50 Train Acc: 0.8200 | Val Acc: 0.4332
[50枚パターン] Epoch 3/50 Train Acc: 0.7400 | Val Acc: 0.4241
[50枚パターン] Epoch 4/50 Train Acc: 0.8000 | Val Acc: 0.5111
[50枚パターン] Epoch 5/50 Train Acc: 0.8200 | Val Acc: 0.6943
[50枚パターン] Epoch 6/50 Train Acc: 0.8800 | Val Acc: 0.6508
[50枚パターン] Epoch 7/50 Train Acc: 0.9000 | Val Acc: 0.6994
[50枚パターン] Epoch 8/50 Train Acc: 0.9800 | Val Acc: 0.6690
[50枚パターン] Epoch 9/50 Train Acc: 0.9600 | Val Acc: 0.6812
[50枚パターン] Epoch 10/50 Train Acc: 1.0000 | Val Acc: 0.7399
[50枚パターン] Epoch 11/50 Train Acc: 0.9200 | Val Acc: 0.6579
[50枚パターン] Epoch 12/50 Train Acc: 0.9000 | Val Acc: 0.6903
[50枚パターン] Epoch 13/50 Train Acc: 0.9200 | Val Acc: 0.7368
[50枚パターン] Epoch 14/50 Train Acc: 0.9600 | Val Acc: 0.7723
[50枚パターン] Epoch 15/50 Train Acc: 0.9400 | Val Acc: 0.7166
[50枚パターン] Epoch 16/50 Train Acc: 0.8000 | Val Acc: 0.6974
[50枚パターン] Epoch 17/50 Train Acc: 0.9200 | Val Acc: 0.4889
[50枚パターン] Epoch 18/50 T

In [7]:
import torch
import torch.nn as nn
from torchvision import datasets, models, transforms
from torch.utils.data import DataLoader, Subset
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import os

def main():
    # -----------------------------
    # 1. 設定
    # -----------------------------
    DATA_DIR = '/home/data/1021_fasttest/crop'
    TRAIN_SIZES = [50, 100, 150, 200]
    NUM_CLASSES = 3
    BATCH_SIZE = 32
    
    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    print(f"評価デバイス: {device}")

    # -----------------------------
    # 2. データの準備 (共通)
    # -----------------------------
    data_transforms = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])
    full_dataset = datasets.ImageFolder(DATA_DIR, transform=data_transforms)

    # 結果を保存するリスト
    results = []

    # -----------------------------
    # 3. ループで順にテスト
    # -----------------------------
    for size in TRAIN_SIZES:
        # パス設定（ユーザーのコードに合わせて crop/ を付与）
        split_file = f'crop_50/dataset_split_{size}.pth'
        model_file = f'crop_50/resnet50_{size}.pth'

        if not os.path.exists(split_file) or not os.path.exists(model_file):
            print(f"スキップ: {size}枚のファイルが見つかりません。")
            continue

        print(f"\n##################################################")
        print(f"  評価開始: 学習データ {size} 枚モデル")
        print(f"##################################################")

        # (1) 分割情報のロード
        split_info = torch.load(split_file, weights_only=False)
        val_indices = split_info['val_indices']
        class_names = split_info['class_names'] # クラス名を取得
        
        # 検証セットを復元
        val_dataset = Subset(full_dataset, val_indices)
        val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

        # (2) モデルのロード
        model = models.resnet50(weights=None)
        model.fc = nn.Linear(model.fc.in_features, NUM_CLASSES)
        model.load_state_dict(torch.load(model_file, map_location=device))
        model = model.to(device)
        model.eval()

        # (3) 推論
        all_preds = []
        all_labels = []
        
        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                _, preds = torch.max(outputs, 1)
                
                all_preds.extend(preds.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())

        # (4) 詳細レポート出力
        print(f"\n--- Classification Report (Size: {size}) ---")
        report = classification_report(all_labels, all_preds, target_names=class_names)
        print(report)

        # (5) 混同行列の作成と保存
        cm = confusion_matrix(all_labels, all_preds)
        cm_df = pd.DataFrame(cm, index=class_names, columns=class_names)

        plt.figure(figsize=(8, 6))
        sns.heatmap(cm_df, annot=True, fmt='d', cmap='Blues')
        plt.title(f'Confusion Matrix (Train Size: {size})')
        plt.ylabel('True Label')
        plt.xlabel('Predicted Label')
        plt.tight_layout()
        
        # 画像として保存
        save_img_name = f'confusion_matrix_{size}.png'
        plt.savefig(save_img_name)
        plt.close() # メモリ解放
        print(f" >> 混同行列を保存しました: {save_img_name}")

        # (6) 結果リストに追加 (AccuracyだけでなくF1スコアも追加)
        acc = accuracy_score(all_labels, all_preds)
        f1 = f1_score(all_labels, all_preds, average='macro')
        
        results.append({
            "Train Size": size,
            "Accuracy": acc,
            "Macro F1": f1,
            "Val Data Count": len(val_indices)
        })

    # -----------------------------
    # 4. 結果まとめ表示
    # -----------------------------
    print("\n\n=== データ数ごとの精度比較まとめ ===")
    df = pd.DataFrame(results)
    # 見やすいように丸める
    pd.options.display.float_format = '{:.4f}'.format
    print(df.to_string(index=False))

    # CSVにも保存
    df.to_csv("accuracy_comparison_detail.csv", index=False)
    print("\nまとめ結果を 'accuracy_comparison_detail.csv' に保存しました。")

if __name__ == '__main__':
    main()

評価デバイス: cpu

##################################################
  評価開始: 学習データ 50 枚モデル
##################################################

--- Classification Report (Size: 50) ---
              precision    recall  f1-score   support

           A       0.49      1.00      0.66       479
           B       0.00      0.00      0.00       285
           C       0.60      0.01      0.03       224

    accuracy                           0.49       988
   macro avg       0.36      0.34      0.23       988
weighted avg       0.37      0.49      0.32       988

 >> 混同行列を保存しました: confusion_matrix_50.png

##################################################
  評価開始: 学習データ 100 枚モデル
##################################################


/usr/local/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))



--- Classification Report (Size: 100) ---
              precision    recall  f1-score   support

           A       1.00      0.40      0.57       462
           B       0.41      0.63      0.50       269
           C       0.43      0.71      0.54       207

    accuracy                           0.54       938
   macro avg       0.61      0.58      0.54       938
weighted avg       0.71      0.54      0.54       938

 >> 混同行列を保存しました: confusion_matrix_100.png

##################################################
  評価開始: 学習データ 150 枚モデル
##################################################

--- Classification Report (Size: 150) ---
              precision    recall  f1-score   support

           A       0.95      0.99      0.97       446
           B       0.81      0.71      0.75       252
           C       0.74      0.79      0.77       190

    accuracy                           0.87       888
   macro avg       0.83      0.83      0.83       888
weighted avg       0.87      0.87      

学習済みじゃないやつ

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, models, transforms
from torch.utils.data import DataLoader, Subset
import os
from collections import Counter
import numpy as np
import math

def main():
    # -----------------------------
    # 1. 設定パラメータ
    # -----------------------------
    DATA_DIR = '/home/data/1021_fasttest/crop'
    BATCH_SIZE = 8
    NUM_EPOCHS = 100
    LEARNING_RATE = 0.001
    NUM_CLASSES = 3
    
    # 学習させたいデータ総数のリスト
    TRAIN_SIZES = [10,20,30,40,50,60,70,80,90, 100,110,120,130,140, 150,160,170,180,190, 200]

    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    print(f"使用デバイス: {device}")

    # -----------------------------
    # 2. データの前処理と読み込み
    # -----------------------------
    data_transforms = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])

    full_dataset = datasets.ImageFolder(DATA_DIR, transform=data_transforms)
    class_names = full_dataset.classes
    all_targets = np.array(full_dataset.targets)
    
    print(f"検知されたクラス: {class_names}") 

    # =========================================================
    #  ここからサイズごとのループ実行
    # =========================================================
    for total_train_size in TRAIN_SIZES:
        print(f"\n##################################################")
        print(f"  開始(Scratch): 学習データ数 {total_train_size} 枚")
        print(f"##################################################")

        # ---------------------------------------------------------
        # クラスごとの枚数を計算
        # ---------------------------------------------------------
        base_count = total_train_size // NUM_CLASSES
        remainder = total_train_size % NUM_CLASSES
        
        target_counts = {}
        for i in range(NUM_CLASSES):
            add = 1 if i < remainder else 0
            target_counts[i] = base_count + add

        print(f"目標内訳: {target_counts}")

        # ---------------------------------------------------------
        # データ分割処理
        # ---------------------------------------------------------
        train_indices = []
        val_indices = []
        
        # ★重要: 前回の実験と比較するため、シードは同じ42にして同じ画像を選ぶようにする
        np.random.seed(42)

        for class_idx in range(NUM_CLASSES):
            indices_of_class = np.where(all_targets == class_idx)[0]
            np.random.shuffle(indices_of_class)
            
            count = target_counts[class_idx]
            
            if len(indices_of_class) < count:
                 raise ValueError(f"クラス {class_names[class_idx]} データ不足")

            train_indices.extend(indices_of_class[:count])
            val_indices.extend(indices_of_class[count:])
        
        train_dataset = Subset(full_dataset, train_indices)
        val_dataset = Subset(full_dataset, val_indices)

        # ---------------------------------------------------------
        # 分割情報の保存 (ファイル名を _scratch 付きに変更)
        # ---------------------------------------------------------
        # ※比較のためデータ分割は同じはずですが、念のため保存します
        split_filename = f'nostudy_crop/dataset_split_scratch_{total_train_size}.pth'
        split_info = {
            'train_indices': train_indices,
            'val_indices': val_indices,
            'class_names': class_names,
            'total_size': total_train_size
        }
        torch.save(split_info, split_filename)

        train_size_real = len(train_dataset)
        val_size_real = len(val_dataset)
        
        # DataLoader作成
        train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
        val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

        # -----------------------------
        # 3. モデルの構築 (【変更点】重みなし)
        # -----------------------------
        # weights=None を指定することで、ランダムな初期値からスタートします
        model = models.resnet50(weights=None)
        
        num_ftrs = model.fc.in_features
        model.fc = nn.Linear(num_ftrs, NUM_CLASSES)
        model = model.to(device)

        # -----------------------------
        # 4. 損失関数と最適化手法
        # -----------------------------
        criterion = nn.CrossEntropyLoss()
        optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

        # -----------------------------
        # 5. 学習ループ
        # -----------------------------
        for epoch in range(NUM_EPOCHS):
            model.train()
            running_loss = 0.0
            correct = 0
            total = 0

            # Training
            for inputs, labels in train_loader:
                inputs, labels = inputs.to(device), labels.to(device)

                optimizer.zero_grad()
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                loss.backward()
                optimizer.step()

                running_loss += loss.item() * inputs.size(0)
                _, predicted = torch.max(outputs, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()

            epoch_loss = running_loss / train_size_real
            epoch_acc = correct / total

            # Validation
            model.eval()
            val_loss = 0.0
            val_correct = 0
            val_total = 0
            
            with torch.no_grad():
                for inputs, labels in val_loader:
                    inputs, labels = inputs.to(device), labels.to(device)
                    outputs = model(inputs)
                    loss = criterion(outputs, labels)
                    
                    val_loss += loss.item() * inputs.size(0)
                    _, predicted = torch.max(outputs, 1)
                    val_total += labels.size(0)
                    val_correct += (predicted == labels).sum().item()

            val_epoch_loss = val_loss / val_size_real
            val_epoch_acc = val_correct / val_total
            
            print(f'[{total_train_size}枚-Scratch] Epoch {epoch+1}/{NUM_EPOCHS} '
                  f'Train Acc: {epoch_acc:.4f} | Val Acc: {val_epoch_acc:.4f}')

        # -----------------------------
        # 6. モデルの保存 (ファイル名を _scratch 付きに変更)
        # -----------------------------
        model_filename = f'nostudy_crop/resnet50_scratch_{total_train_size}.pth'
        torch.save(model.state_dict(), model_filename)
        print(f"★ モデル保存完了: {model_filename}\n")

    print("全てのパターンの学習(Scratch)が完了しました。")

if __name__ == '__main__':
    main()

使用デバイス: cpu
検知されたクラス: ['A', 'B', 'C']

##################################################
  開始(Scratch): 学習データ数 50 枚
##################################################
目標内訳: {0: 17, 1: 17, 2: 16}
[50枚-Scratch] Epoch 1/50 Train Acc: 0.2800 | Val Acc: 0.4858
[50枚-Scratch] Epoch 2/50 Train Acc: 0.4800 | Val Acc: 0.2885
[50枚-Scratch] Epoch 3/50 Train Acc: 0.3200 | Val Acc: 0.2885
[50枚-Scratch] Epoch 4/50 Train Acc: 0.6000 | Val Acc: 0.2267
[50枚-Scratch] Epoch 5/50 Train Acc: 0.6600 | Val Acc: 0.2267
[50枚-Scratch] Epoch 6/50 Train Acc: 0.8200 | Val Acc: 0.2470
[50枚-Scratch] Epoch 7/50 Train Acc: 0.9600 | Val Acc: 0.2470
[50枚-Scratch] Epoch 8/50 Train Acc: 0.9800 | Val Acc: 0.2611
[50枚-Scratch] Epoch 9/50 Train Acc: 0.9600 | Val Acc: 0.3451
[50枚-Scratch] Epoch 10/50 Train Acc: 0.9800 | Val Acc: 0.3745
[50枚-Scratch] Epoch 11/50 Train Acc: 1.0000 | Val Acc: 0.3856
[50枚-Scratch] Epoch 12/50 Train Acc: 1.0000 | Val Acc: 0.3421
[50枚-Scratch] Epoch 13/50 Train Acc: 1.0000 | Val Acc: 0.3229
[50枚-Sc

In [ ]:
import torch
import torch.nn as nn
from torchvision import datasets, models, transforms
from torch.utils.data import DataLoader, Subset
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import os

def main():
    # -----------------------------
    # 1. 設定
    # -----------------------------
    DATA_DIR = '/home/data/1021_fasttest/crop'
    TRAIN_SIZES = [10,20,30,40,50,60,70,80,90, 100,110,120,130,140, 150,160,170,180,190, 200]
    NUM_CLASSES = 3
    BATCH_SIZE = 8
    
    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    print(f"評価デバイス: {device}")

    # -----------------------------
    # 2. データの準備 (共通)
    # -----------------------------
    data_transforms = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])
    full_dataset = datasets.ImageFolder(DATA_DIR, transform=data_transforms)

    # 結果を保存するリスト
    results = []

    # -----------------------------
    # 3. ループで順にテスト
    # -----------------------------
    for size in TRAIN_SIZES:
        # パス設定（ユーザーのコードに合わせて crop/ を付与）
        split_file = f'/home/src/deeplerning_model/nostudy_crop/dataset_split_scratch_{size}.pth'
        model_file = f'/home/src/deeplerning_model/nostudy_crop/resnet50_scratch_{size}.pth'
        # split_file = f'/home/src/deeplerning_model/res18_crop/dataset_split_{size}.pth'
        # model_file = f'/home/src/deeplerning_model/res18_crop/resnet18_{size}.pth'

        if not os.path.exists(split_file) or not os.path.exists(model_file):
            print(f"スキップ: {size}枚のファイルが見つかりません。")
            continue

        print(f"\n##################################################")
        print(f"  評価開始: 学習データ {size} 枚モデル")
        print(f"##################################################")

        # (1) 分割情報のロード
        split_info = torch.load(split_file, weights_only=False)
        val_indices = split_info['val_indices']
        class_names = split_info['class_names'] # クラス名を取得
        
        # 検証セットを復元
        val_dataset = Subset(full_dataset, val_indices)
        val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

        # (2) モデルのロード
        model = models.resnet50(weights=None)
        model.fc = nn.Linear(model.fc.in_features, NUM_CLASSES)
        model.load_state_dict(torch.load(model_file, map_location=device))
        model = model.to(device)
        model.eval()

        # (3) 推論
        all_preds = []
        all_labels = []
        
        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                _, preds = torch.max(outputs, 1)
                
                all_preds.extend(preds.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())

        # (4) 詳細レポート出力
        print(f"\n--- Classification Report (Size: {size}) ---")
        report = classification_report(all_labels, all_preds, target_names=class_names)
        print(report)

        # (5) 混同行列の作成と保存
        cm = confusion_matrix(all_labels, all_preds)
        cm_df = pd.DataFrame(cm, index=class_names, columns=class_names)

        plt.figure(figsize=(8, 6))
        sns.heatmap(cm_df, annot=True, fmt='d', cmap='Blues')
        plt.title(f'Confusion Matrix (Train Size: {size})')
        plt.ylabel('True Label')
        plt.xlabel('Predicted Label')
        plt.tight_layout()
        
        # 画像として保存
        save_img_name = f'confusion_matrix_{size}.png'
        plt.savefig(save_img_name)
        plt.close() # メモリ解放
        print(f" >> 混同行列を保存しました: {save_img_name}")

        # (6) 結果リストに追加 (AccuracyだけでなくF1スコアも追加)
        acc = accuracy_score(all_labels, all_preds)
        f1 = f1_score(all_labels, all_preds, average='macro')
        
        results.append({
            "Train Size": size,
            "Accuracy": acc,
            "Macro F1": f1,
            "Val Data Count": len(val_indices)
        })

    # -----------------------------
    # 4. 結果まとめ表示
    # -----------------------------
    print("\n\n=== データ数ごとの精度比較まとめ ===")
    df = pd.DataFrame(results)
    # 見やすいように丸める
    pd.options.display.float_format = '{:.4f}'.format
    print(df.to_string(index=False))

    # CSVにも保存
    df.to_csv("accuracy_comparison_detail.csv", index=False)
    print("\nまとめ結果を 'accuracy_comparison_detail.csv' に保存しました。")

if __name__ == '__main__':
    main()

評価デバイス: cuda:0

##################################################
  評価開始: 学習データ 10 枚モデル
##################################################

--- Classification Report (Size: 10) ---
              precision    recall  f1-score   support

           A       0.75      0.28      0.40       492
           B       0.26      0.38      0.31       299
           C       0.36      0.62      0.46       237

    accuracy                           0.39      1028
   macro avg       0.46      0.43      0.39      1028
weighted avg       0.52      0.39      0.39      1028

 >> 混同行列を保存しました: confusion_matrix_10.png

##################################################
  評価開始: 学習データ 20 枚モデル
##################################################

--- Classification Report (Size: 20) ---
              precision    recall  f1-score   support

           A       0.82      0.81      0.81       489
           B       0.46      0.37      0.41       295
           C       0.53      0.67      0.59       234

    accurac

/usr/local/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [12]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, models, transforms
from torch.utils.data import DataLoader, Subset
import os
import numpy as np

def main():
    # -----------------------------
    # 1. 設定パラメータ
    # -----------------------------
    DATA_DIR = '/home/data/1021_fasttest/crop'
    BATCH_SIZE = 8
    NUM_EPOCHS = 10
    LEARNING_RATE = 0.001
    NUM_CLASSES = 3
    
    # クラスごとの使用上限枚数（C等級に合わせて240枚）
    MAX_PER_CLASS = 240
    # 評価用に固定する枚数（各クラス）
    VAL_FIXED_NUM = 170
    # 学習用に使える最大枚数（240 - 170 = 70枚）
    TRAIN_POOL_NUM = MAX_PER_CLASS - VAL_FIXED_NUM  # = 70

    # 学習させたいデータ総数のリスト
    # 最大値は 70枚 * 3クラス = 210枚 まで
    # TRAIN_SIZES = [ 50, 60, 70, 80, 90, 100, 110, 120, 130, 140, 150, 160, 170, 180, 190, 200]
    TRAIN_SIZES = [60]

    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    print(f"使用デバイス: {device}")

    # -----------------------------
    # 2. データの前処理と読み込み
    # -----------------------------
    data_transforms = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])

    full_dataset = datasets.ImageFolder(DATA_DIR, transform=data_transforms)
    class_names = full_dataset.classes
    all_targets = np.array(full_dataset.targets)
    
    print(f"検知されたクラス: {class_names}") 

    # 保存先ディレクトリ
    os.makedirs('nostudy_crop', exist_ok=True)

    # =========================================================
    #  【重要】データの固定分割処理
    # =========================================================
    # ここで評価データと学習用プールを確定させます
    
    # 各クラスの学習用候補インデックス（最大70枚）を格納する辞書
    train_pool_indices_map = {i: [] for i in range(NUM_CLASSES)}
    # 固定された評価用インデックス（全クラス分合計）
    fixed_val_indices = []

    np.random.seed(42) # シード固定

    print("\n--- データセットの分割準備 ---")
    for class_idx in range(NUM_CLASSES):
        # そのクラスの全画像インデックスを取得
        indices_of_class = np.where(all_targets == class_idx)[0]
        
        # データ数チェック
        if len(indices_of_class) < MAX_PER_CLASS:
            raise ValueError(f"クラス {class_names[class_idx]} の枚数が {len(indices_of_class)}枚しかありません。{MAX_PER_CLASS}枚必要です。")
        
        # ランダムにシャッフル
        np.random.shuffle(indices_of_class)
        
        # 240枚だけピックアップ
        selected_indices = indices_of_class[:MAX_PER_CLASS]
        
        # 前半70枚を学習用プール、後半170枚を評価用に固定
        train_pool = selected_indices[:TRAIN_POOL_NUM]
        val_fixed = selected_indices[TRAIN_POOL_NUM:]
        
        train_pool_indices_map[class_idx] = train_pool
        fixed_val_indices.extend(val_fixed)
        
        print(f"[{class_names[class_idx]}] 学習プール: {len(train_pool)}枚, 固定評価: {len(val_fixed)}枚")

    # 評価セットは固定なのでここでDataset化（ループ内で変わりません）
    val_dataset = Subset(full_dataset, fixed_val_indices)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
    print(f"評価用データ総数: {len(val_dataset)}枚 (固定)")
    print("------------------------------\n")

    # =========================================================
    #  サイズごとのループ実行
    # =========================================================
    for total_train_size in TRAIN_SIZES:
        print(f"##################################################")
        print(f"  開始(Swin-T Scratch): 学習データ数 {total_train_size} 枚")
        print(f"##################################################")

        # ---------------------------------------------------------
        # 今回のループで使用する学習データをプールから取り出す
        # ---------------------------------------------------------
        base_count = total_train_size // NUM_CLASSES
        remainder = total_train_size % NUM_CLASSES
        
        current_train_indices = []

        for class_idx in range(NUM_CLASSES):
            # 今回このクラスから何枚使うか計算
            count = base_count + (1 if class_idx < remainder else 0)
            
            # 学習プールから先頭count枚を取得
            # ※プールは既にシャッフル済みなので、先頭から取ればランダム性は保たれます
            indices = train_pool_indices_map[class_idx][:count]
            
            if len(indices) < count:
                raise ValueError(f"学習プール不足: クラス{class_idx}に{count}枚要求されましたが{len(indices)}枚しかありません。")
            
            current_train_indices.extend(indices)
        
        # 今回の学習セット作成
        train_dataset = Subset(full_dataset, current_train_indices)
        train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)

        # ---------------------------------------------------------
        # 分割情報の保存
        # ---------------------------------------------------------
        split_filename = f'res50_test/dataset_split_{total_train_size}.pth'
        split_info = {
            'train_indices': current_train_indices,
            'val_indices': fixed_val_indices, # 常に同じデータが入ります
            'class_names': class_names,
            'total_size': total_train_size
        }
        torch.save(split_info, split_filename)

        train_size_real = len(train_dataset)
        val_size_real = len(val_dataset)

        # -----------------------------
        # 3. モデルの構築 (ResNet50)
        # -----------------------------
        model = models.resnet50(weights=None)
        num_ftrs = model.fc.in_features
        model.fc = nn.Linear(num_ftrs, NUM_CLASSES)
        model = model.to(device)

        # -----------------------------
        # 4. 損失関数と最適化手法
        # -----------------------------
        criterion = nn.CrossEntropyLoss()
        optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

        # -----------------------------
        # 5. 学習ループ
        # -----------------------------
        for epoch in range(NUM_EPOCHS):
            model.train()
            running_loss = 0.0
            correct = 0
            total = 0

            # Training
            for inputs, labels in train_loader:
                inputs, labels = inputs.to(device), labels.to(device)

                optimizer.zero_grad()
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                loss.backward()
                optimizer.step()

                running_loss += loss.item() * inputs.size(0)
                _, predicted = torch.max(outputs, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()

            epoch_loss = running_loss / train_size_real
            epoch_acc = correct / total

            # Validation (固定されたデータセットで評価)
            model.eval()
            val_loss = 0.0
            val_correct = 0
            val_total = 0
            
            with torch.no_grad():
                for inputs, labels in val_loader:
                    inputs, labels = inputs.to(device), labels.to(device)
                    outputs = model(inputs)
                    loss = criterion(outputs, labels)
                    
                    val_loss += loss.item() * inputs.size(0)
                    _, predicted = torch.max(outputs, 1)
                    val_total += labels.size(0)
                    val_correct += (predicted == labels).sum().item()

            val_epoch_loss = val_loss / val_size_real
            val_epoch_acc = val_correct / val_total
            
            print(f'[{total_train_size}枚] Ep {epoch+1}/{NUM_EPOCHS} '
                  f'Train Acc: {epoch_acc:.4f} | Val Acc: {val_epoch_acc:.4f}')

        # -----------------------------
        # 6. モデルの保存
        # -----------------------------
        model_filename = f'res50_test/resnet50_{total_train_size}.pth'
        torch.save(model.state_dict(), model_filename)
        print(f"★ モデル保存完了: {model_filename}\n")

    print("全てのパターンの学習(ResNet50 Scratch)が完了しました。")
if __name__ == '__main__':
    main()

使用デバイス: cuda:0
検知されたクラス: ['A', 'B', 'C']

--- データセットの分割準備 ---
[A] 学習プール: 70枚, 固定評価: 170枚
[B] 学習プール: 70枚, 固定評価: 170枚
[C] 学習プール: 70枚, 固定評価: 170枚
評価用データ総数: 510枚 (固定)
------------------------------

##################################################
  開始(Swin-T Scratch): 学習データ数 60 枚
##################################################
[60枚] Ep 1/10 Train Acc: 0.3167 | Val Acc: 0.3333
[60枚] Ep 2/10 Train Acc: 0.5667 | Val Acc: 0.5157
[60枚] Ep 3/10 Train Acc: 0.5167 | Val Acc: 0.4902
[60枚] Ep 4/10 Train Acc: 0.5167 | Val Acc: 0.5373
[60枚] Ep 5/10 Train Acc: 0.7167 | Val Acc: 0.5824
[60枚] Ep 6/10 Train Acc: 0.7333 | Val Acc: 0.4961
[60枚] Ep 7/10 Train Acc: 0.8167 | Val Acc: 0.5784
[60枚] Ep 8/10 Train Acc: 0.7333 | Val Acc: 0.5451
[60枚] Ep 9/10 Train Acc: 0.8333 | Val Acc: 0.6784
[60枚] Ep 10/10 Train Acc: 0.7833 | Val Acc: 0.6784
★ モデル保存完了: res50_test/resnet50_60.pth

全てのパターンの学習(ResNet50 Scratch)が完了しました。


In [13]:
import torch
import torch.nn as nn
from torchvision import datasets, models, transforms
from torch.utils.data import DataLoader, Subset
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import os

def main():
    # -----------------------------
    # 1. 設定
    # -----------------------------
    DATA_DIR = '/home/data/1021_fasttest/crop'
    SAVE_DIR = 'res50_test'  # 学習時に保存したフォルダ
    
    # 学習データのサイズリスト (学習コードと同じもの)
    # TRAIN_SIZES = [50, 60, 70, 80, 90, 100, 110, 120, 130, 140, 150, 160, 170, 180, 190, 200]
    TRAIN_SIZES = [60]

    
    NUM_CLASSES = 3
    BATCH_SIZE = 8
    
    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    print(f"評価デバイス: {device}")

    # -----------------------------
    # 2. データの準備 (共通)
    # -----------------------------
    data_transforms = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])
    full_dataset = datasets.ImageFolder(DATA_DIR, transform=data_transforms)

    # 結果を保存するリスト
    results = []
    
    # 保存先ディレクトリがない場合は作成（念のため）
    os.makedirs(SAVE_DIR, exist_ok=True)

    # -----------------------------
    # 3. ループで順にテスト
    # -----------------------------
    for size in TRAIN_SIZES:
        # パス設定（Swin用のファイル名に変更）
        split_file = os.path.join(SAVE_DIR, f'dataset_split_{size}.pth')
        model_file = os.path.join(SAVE_DIR, f'resnet50_{size}.pth')

        if not os.path.exists(split_file) or not os.path.exists(model_file):
            print(f"スキップ: {size}枚のファイルが見つかりません。")
            continue

        print(f"\n##################################################")
        print(f"  評価開始(Swin): 学習データ {size} 枚モデル")
        print(f"##################################################")

        # (1) 分割情報のロード
        # 学習コードで「評価データは固定」されていますが、そのインデックス情報は各splitファイルに含まれています
        try:
            split_info = torch.load(split_file, weights_only=False)
        except TypeError:
            split_info = torch.load(split_file)

        val_indices = split_info['val_indices']
        class_names = split_info['class_names'] # クラス名を取得
        
        # 検証セットを復元
        val_dataset = Subset(full_dataset, val_indices)
        val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

        # (2) モデルのロード 
        model = models.resnet50(weights=None)

        num_ftrs = model.fc.in_features
        model.fc = nn.Linear(num_ftrs, NUM_CLASSES)
        
        # 重みの読み込み
        model.load_state_dict(torch.load(model_file, map_location=device))
        model = model.to(device)
        model.eval()

        # (3) 推論
        all_preds = []
        all_labels = []
        
        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                _, preds = torch.max(outputs, 1)
                
                all_preds.extend(preds.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())

        # (4) 詳細レポート出力
        print(f"\n--- Classification Report (Size: {size}) ---")
        # zero_division=0: 未予測クラスがあっても警告を出さない設定
        report = classification_report(all_labels, all_preds, target_names=class_names, zero_division=0)
        print(report)

        # (5) 混同行列の作成と保存
        cm = confusion_matrix(all_labels, all_preds)
        cm_df = pd.DataFrame(cm, index=class_names, columns=class_names)

        plt.figure(figsize=(8, 6))
        sns.heatmap(cm_df, annot=True, fmt='d', cmap='Blues')
        plt.title(f'Confusion Matrix (Swin Train Size: {size})')
        plt.ylabel('True Label')
        plt.xlabel('Predicted Label')
        plt.tight_layout()
        
        # 画像として保存 (swin用の名前に変更)
        save_img_name = os.path.join(SAVE_DIR, f'swin_confusion_matrix_{size}.png')
        plt.savefig(save_img_name)
        plt.close() # メモリ解放
        print(f" >> 混同行列を保存しました: {save_img_name}")

        # (6) 結果リストに追加
        acc = accuracy_score(all_labels, all_preds)
        f1 = f1_score(all_labels, all_preds, average='macro')
        
        results.append({
            "Train Size": size,
            "Accuracy": acc,
            "Macro F1": f1,
            "Val Data Count": len(val_indices)
        })

    # -----------------------------
    # 4. 結果まとめ表示
    # -----------------------------
    print("\n\n=== [Swin] データ数ごとの精度比較まとめ ===")
    df = pd.DataFrame(results)
    
    # 見やすいように丸める
    pd.options.display.float_format = '{:.4f}'.format
    print(df.to_string(index=False))

    # CSVにも保存
    csv_save_path = os.path.join(SAVE_DIR, "swin_accuracy_comparison_detail.csv")
    df.to_csv(csv_save_path, index=False)
    print(f"\nまとめ結果を '{csv_save_path}' に保存しました。")

if __name__ == '__main__':
    main()

評価デバイス: cuda:0

##################################################
  評価開始(Swin): 学習データ 60 枚モデル
##################################################

--- Classification Report (Size: 60) ---
              precision    recall  f1-score   support

           A       0.85      0.89      0.87       170
           B       0.66      0.22      0.33       170
           C       0.57      0.92      0.71       170

    accuracy                           0.68       510
   macro avg       0.69      0.68      0.64       510
weighted avg       0.69      0.68      0.64       510

 >> 混同行列を保存しました: res50_test/swin_confusion_matrix_60.png


=== [Swin] データ数ごとの精度比較まとめ ===
 Train Size  Accuracy  Macro F1  Val Data Count
         60    0.6784    0.6364             510

まとめ結果を 'res50_test/swin_accuracy_comparison_detail.csv' に保存しました。
